In [1]:
import numpy as np
import pandas as pd
from numpy import loadtxt
from numpy import exp


from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import accuracy_score,mean_absolute_error,mean_squared_error
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from keras.wrappers.scikit_learn import KerasRegressor,KerasClassifier
from torch.autograd import Variable

import torch
from torch import nn
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.utils.data import Dataset, TensorDataset
import torch.nn.functional as F

%matplotlib inline

pd.set_option('max_rows',999)

#### detect device

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

### Import data

In [3]:
df_sunspots = pd.read_csv('data/data.csv.txt')

In [45]:
# Split month

df_sunspots.columns = ['Data','Sunspots']
df_sunspots['Month'] = df_sunspots.Data.apply(lambda x: int(x.split('-')[1]))

#build phase space
m = 10
df_phase=pd.DataFrame([df_sunspots.Sunspots.iloc[-m+d:d].values for d in range(df_sunspots.shape[0])])
df_phase.columns =['Sunspots_'+str(col-10) for col in df_phase.columns]
df_target = df_sunspots['Sunspots'].iloc[m:]

#get month dummies 
df_data=pd.concat([pd.get_dummies(df_sunspots.Month,prefix='Month_'),df_phase],axis=1).iloc[m:]
df_data.set_index(df_sunspots['Data'].iloc[m:],inplace=True)

### Train split

In [5]:
X=df_data.values
y=df_target.values

In [6]:
tscv = TimeSeriesSplit(n_splits=2,test_size=30)

In [7]:
for train_index, test_index in tscv.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

In [8]:
tscv = TimeSeriesSplit(n_splits=2,test_size=30)

In [9]:
for train_index, test_index in tscv.split(X_train):
    X_train_2, X_valid = X_train[train_index], X_train[test_index]
    y_train_2, y_valid = y_train[train_index], y_train[test_index]

#### To tensor

In [10]:
x_train_tensor = torch.from_numpy(X_train_2).float().to(device)
y_train_tensor = torch.from_numpy(y_train_2).float().to(device)

x_val_tensor = torch.from_numpy(X_valid).float().to(device)
y_val_tensor = torch.from_numpy(y_valid).float().to(device)

train_data = TensorDataset(x_train_tensor, y_train_tensor)
train_loader = DataLoader(dataset=train_data, batch_size=10)

val_data = TensorDataset(x_val_tensor, y_val_tensor)
val_loader = DataLoader(dataset=val_data, batch_size=20)

## Define Model and Parameters

In [42]:
def exp_decay(self, epoch):
    initial_lrate = 0.001
    k = 0.1
    lrate = initial_lrate * exp(-k*epoch)
    return lrate

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        
        # define an MLP layers
        self.fc1 = nn.Linear(in_features=22, out_features=128)
        self.fc2 = nn.Linear(in_features=128, out_features=128)
        
        self.fc3 = nn.Linear(in_features=128, out_features=128)
        self.fc4 = nn.Linear(in_features=128, out_features=128)
        self.fc5 = nn.Linear(in_features=128, out_features=128)
        self.fc6 = nn.Linear(in_features=128, out_features=128)
        self.fc7 = nn.Linear(in_features=128, out_features=128)
        self.fc8 = nn.Linear(in_features=128, out_features=32)
        self.output = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))
        x = F.relu(self.fc6(x))
        x = F.relu(self.fc7(x))
        x = F.relu(self.fc8(x))
        x = self.output(x)
        return x


In [43]:
model = MLP().to(device)
model

MLP(
  (fc1): Linear(in_features=22, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=128, bias=True)
  (fc5): Linear(in_features=128, out_features=128, bias=True)
  (fc6): Linear(in_features=128, out_features=128, bias=True)
  (fc7): Linear(in_features=128, out_features=128, bias=True)
  (fc8): Linear(in_features=128, out_features=32, bias=True)
  (output): Linear(in_features=32, out_features=1, bias=True)
)

In [33]:
torch.__version__

'1.11.0+cpu'

In [13]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


epochs = 400

## Train Model

In [44]:
%%time

losses = []
val_losses = []

for i in range(epochs):
    model.train()
    for x_batch, y_batch in train_loader:
        
        x_batch = x_batch
        y_batch = y_batch
        # Makes predictions
        yhat = model.forward(x_batch)
        # Computes loss
        loss = criterion(y_batch, yhat)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        
    model.eval()      
    with torch.no_grad():
        for x_val, y_val in val_loader:
            x_val = x_val
            y_val = y_val



            yhat = model.forward(x_val)
            #print("Predição:",yhat)
            val_loss = criterion(y_val, yhat)
            val_losses.append(val_loss.item())
    print('Epoch: %.2f Loss: %.2f - Loss_vald: %.2f' % (i,np.mean(losses),np.mean(val_losses)) )          


Epoch: 0.00 Loss: 4219.07 - Loss_vald: 22242.84
Epoch: 1.00 Loss: 4219.07 - Loss_vald: 22242.84
Epoch: 2.00 Loss: 4219.07 - Loss_vald: 22242.84
Epoch: 3.00 Loss: 4219.07 - Loss_vald: 22242.84
Epoch: 4.00 Loss: 4219.07 - Loss_vald: 22242.84
Epoch: 5.00 Loss: 4219.07 - Loss_vald: 22242.84
Epoch: 6.00 Loss: 4219.07 - Loss_vald: 22242.84
Epoch: 7.00 Loss: 4219.07 - Loss_vald: 22242.84
Epoch: 8.00 Loss: 4219.07 - Loss_vald: 22242.84
Epoch: 9.00 Loss: 4219.07 - Loss_vald: 22242.84
Wall time: 5.81 s


#### Evaluate

In [15]:
X_test_tensor = torch.from_numpy(X_test).float().to(device)

In [16]:
preds = []
with torch.no_grad():
    for val in X_test_tensor:
        
        y_hat = model.forward(val)
        print(y_hat)
        preds.append(y_hat.item())

tensor([143.1206])
tensor([152.4084])
tensor([167.1030])
tensor([174.1981])
tensor([186.2889])
tensor([176.2456])
tensor([179.7218])
tensor([163.0756])
tensor([180.7944])
tensor([187.1343])
tensor([166.4605])
tensor([138.8980])
tensor([136.0804])
tensor([131.5803])
tensor([131.4777])
tensor([131.2605])
tensor([125.6285])
tensor([115.5147])
tensor([128.5608])
tensor([120.7388])
tensor([103.2284])
tensor([93.4974])
tensor([95.1542])
tensor([101.8664])
tensor([100.6745])
tensor([100.2129])
tensor([93.9887])
tensor([79.4772])
tensor([79.6701])
tensor([69.9275])


In [17]:
print('MAE: %.2f' % mean_absolute_error(y_test,preds))

MAE: 29.52
